
# ERP–Blockchain Adaptive Execution — v5: DESIGN FROZEN + CONFIRMATION ON UNTOUCHED 2021 DATA

**Protocol (this is what makes the evaluation hard to attack):**

| Period | Dates | Use |
|---|---|---|
| Train | 2020-01-01 → 2020-09-30 | fit forecasters |
| Validation | 2020-10-01 → 2020-12-31 | model selection + **all controller tuning** |
| **Held-out test** | **2021-01-01 → 2021-03-31** | **chronologically after all tuning, but re-used across design iterations (v1–v5.2.1) - it is NOT an independent confirmation set** |

The configuration is hashed (`DESIGN FREEZE hash`, printed in cell 1 and saved in `RESULTS_SUMMARY.json`). Earlier exploratory runs (v1–v4) used 2020 only; **state this in the paper** ("controller design was developed on 2020 data; 2021-Q1 is a held-out held-out test period").

**What v5 changes vs v4** (motivated by the v4 *2020* results — disclosed, and now tested on fresh data):
1. **Priority ordering is always on** (it costs nothing); the controller adapts only the *off-chain batching* level (`PRIO` → `MID` → `CONG`). v4's low regime was FIFO, which needlessly hurt HIGH-priority latency.
2. **Decision statistic is a tuned hyper-parameter**: `I` = backlog + expected load of the coming interval, `H` = predicted backlog growth over the next 6 blocks. Chosen on *validation* (epsilon-constraint rule: least off-chain use within 5 % of the best latency objective).
3. Everything else (simulator, baselines, ablations, placebo, sanity tests, drift stress test, claims scorecard) is unchanged, so results are comparable with v4.

> `QUICK=True` = ~5 min smoke test on a small slice (never report QUICK numbers). Full run ≈ 60–75 min on Colab.
>
> **Tip:** to skip ~80 % of the download, upload `api_checkpoint.pkl` (from your `results_v4.zip`) into `/content/results_v5/` *before* running cell 2.


In [ ]:
# 0) INSTALL (Colab)
!pip -q install pyarrow requests

In [ ]:
# 1) IMPORTS + CONFIG
import os, sys, json, time, warnings, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as sps
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings("ignore")

# ---------------- CONFIG ----------------
DATA_SOURCE  = "api"        # "api" = FREE public Bitcoin block API (no account, no key)  |  "bigquery" = needs a Google Cloud project
PROJECT_ID   = ""           # only used if DATA_SOURCE == "bigquery"  (your Google Cloud Project ID)
YEAR_START, YEAR_END = "2020-01-01", "2021-04-01"
MIN_BLOCKS   = 64000                      # 2020-01-01 -> 2021-04-01 has ~66,000 blocks
SEED         = 42
TRAIN_FRAC, VAL_FRAC = 0.60, 0.20         # used ONLY in QUICK mode
T_TRAIN_END, T_VAL_END = "2020-10-01", "2021-01-01"   # train | validation (tuning) | CONFIRMATION TEST = 2021-01-01 .. 2021-03-31
EMBARGO      = 6                          # purge: demand label looks 6 blocks ahead
QUICK        = False
OUT_DIR      = "/content/results_v5_2_1"
os.makedirs(OUT_DIR, exist_ok=True)

# closed loop (walk-forward) on the DEMAND forecaster
CHUNK, WINDOW, PERIODIC_EVERY = 144, 20000, 20
PH_DELTA, PH_LAMBDA = 0.05, 1.0

# simulator (every number below is an explicit, testable assumption)
SIM = dict(
    CAP=100.0, CLASS_P=[0.10, 0.60, 0.30], CLASS_W=np.array([3.0, 1.0, 0.0]),   # HIGH, NORMAL, LOW
    T_AGE=1800.0, ANCHOR=1.0, OFFLOADABLE_P=0.5, MAX_WAIT_PARTIAL=1200.0,
    SIZE_SIGMA=0.8, SLA=3600.0, WARMUP_FRAC=0.05, DT_MEAN=600.0,
)
RHOS, N_SEEDS = [0.5, 0.7, 0.85, 0.95], 5
TUNE_RHOS, TUNE_SEEDS = [0.3, 0.6, 0.9], 1 # tuned on SEVERAL loads so the controller generalises (0.2 and 0.95 are unseen)
SLA_FLOOR = 0.90                           # pre-specified constraint: every class must keep SLA hit-rate >= 90 %
GRID_LO, GRID_HI = [0.0, 0.4, 0.8], [0.6, 1.0, 1.5, 2.5]
SENS_SEEDS = 2
EPS_TOL = 0.05          # eps-constraint: among configs within 5 % of the best latency objective pick the one using the LEAST off-chain
H_FWD = 6               # forecast horizon (blocks) = length of the demand window the forecaster predicts
RECAL_WINDOW, COOLDOWN = 6000, 2   # recalibration trains on the most recent blocks; min cycles between recalibrations
DRIFT_SCALE = 1.35      # drift-injection stress test: workload x1.35 from the middle of the test period
if QUICK:
    RHOS, N_SEEDS, TUNE_SEEDS, SENS_SEEDS, TUNE_RHOS = [0.9], 2, 1, 1, [0.3, 0.9]
    GRID_LO, GRID_HI = [0.0], [0.6, 1.5]
SIM["H"] = H_FWD
CLS = ["HIGH", "NORMAL", "LOW"]
import hashlib
CODE_VERSION = "v5.2.1"   # bump on ANY change that can alter results
NORMALISATION = "demand scale = TRAINING-period mean (fixed reference for validation and confirmation)"
CONFIG_HASH = hashlib.sha256(json.dumps(dict(CODE_VERSION=CODE_VERSION, NORMALISATION=NORMALISATION,
    SIM={k: (v.tolist() if hasattr(v, "tolist") else v) for k, v in SIM.items()}, RHOS=RHOS, N_SEEDS=N_SEEDS, TUNE_RHOS=TUNE_RHOS,
    TUNE_SEEDS=TUNE_SEEDS, GRID_LO=GRID_LO, GRID_HI=GRID_HI, EPS_TOL=EPS_TOL, SLA_FLOOR=SLA_FLOOR, H_FWD=H_FWD, CHUNK=CHUNK,
    WINDOW=WINDOW, RECAL_WINDOW=RECAL_WINDOW, COOLDOWN=COOLDOWN, PH=(PH_DELTA, PH_LAMBDA), EMBARGO=EMBARGO, SEED=SEED,
    SPLIT=(YEAR_START, T_TRAIN_END, T_VAL_END, YEAR_END)), sort_keys=True).encode()).hexdigest()[:16]
print("Config OK | QUICK =", QUICK)
print("CODE VERSION:", CODE_VERSION, "|", NORMALISATION)
print("DESIGN FREEZE hash:", CONFIG_HASH, " <- record it; it must be identical in every run that you report")

In [ ]:
# 2) DATA EXTRACTION + COVERAGE GUARD  (this is what prevents the "15,000 blocks" mistake)
CACHE = f"{OUT_DIR}/blocks_{YEAR_START}_{YEAR_END}.parquet"

# ---------- FREE source: public Esplora-compatible block APIs (no account / no key / no billing) ----------
API_BASES = ["https://mempool.space/api", "https://blockstream.info/api"]
H_LO, H_HI = 610_000, 678_000         # generous height range that contains all of 2020; filtered by timestamp afterwards
if QUICK: H_HI = H_LO + 13_000

def fetch_from_api(workers=4):
    import requests, pickle, concurrent.futures as cf
    starts = list(range(H_HI, H_LO, -10))          # every request returns >= 10 blocks going DOWN from `start` -> full coverage
    ck = f"{OUT_DIR}/api_checkpoint.pkl"
    done = pickle.load(open(ck, "rb")) if os.path.exists(ck) else {}
    todo = [h for h in starts if h not in done]
    print(f"API fetch: {len(starts)} pages total, {len(done)} already cached, {len(todo)} to fetch (resumable)")
    sess = requests.Session(); sess.headers.update({"User-Agent": "research-notebook/1.0"})
    def one(h):
        for attempt in range(8):
            base = API_BASES[(attempt // 2) % len(API_BASES)]     # 2 tries per host, then switch host
            try:
                r = sess.get(f"{base}/blocks/{h}", timeout=30)
                if r.status_code == 200:
                    return h, [(b["height"], b["timestamp"], b["tx_count"], b["size"], b["weight"]) for b in r.json()]
            except Exception:
                pass
            time.sleep(min(30, 2 ** attempt))                     # back off on 429 / network errors
        return h, None
    failed, t0 = 0, time.time()
    with cf.ThreadPoolExecutor(max_workers=workers) as ex:
        for i, (h, rows) in enumerate(ex.map(one, todo), 1):
            if rows is None: failed += 1
            else: done[h] = rows
            if i % 250 == 0:
                pickle.dump(done, open(ck, "wb")); print(f"  {i}/{len(todo)} pages  ({time.time()-t0:.0f}s)")
    pickle.dump(done, open(ck, "wb"))
    if failed:
        raise RuntimeError(f"{failed} pages failed after retries. Just re-run this cell: it RESUMES from the checkpoint. "
                           "If it keeps failing, wait a few minutes (rate limit) or set DATA_SOURCE='bigquery'.")
    rows = [r for v in done.values() for r in v]
    r_ = pd.DataFrame(rows, columns=["number", "ts", "transaction_count", "size", "weight"]).drop_duplicates("number")
    r_["block_timestamp"] = pd.to_datetime(r_["ts"], unit="s", utc=True)
    r_ = r_[(r_.block_timestamp >= pd.Timestamp(YEAR_START, tz="UTC")) & (r_.block_timestamp < pd.Timestamp(YEAR_END, tz="UTC"))]
    r_ = r_.sort_values("number").reset_index(drop=True)[["number", "block_timestamp", "transaction_count", "size", "weight"]]
    r_.to_parquet(CACHE); return r_

# ---------- OPTIONAL source: BigQuery (free tier / sandbox, but needs a Google Cloud project you have access to) ----------
def resolve_project_id():
    pid = (PROJECT_ID or "").strip()
    bad = (not pid) or ("ENTER" in pid.upper()) or ("YOUR" in pid.upper()) or pid.startswith("<")
    if bad:
        try:
            import google.auth
            _, pid = google.auth.default()
        except Exception:
            pid = None
    if not pid:
        pid = input("Paste your Google Cloud PROJECT ID (not the project name): ").strip()
    print("Using BigQuery project:", pid); return pid

def fetch_from_bigquery():
    from google.colab import auth
    auth.authenticate_user()
    import subprocess, sys; subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'google-cloud-bigquery', 'db-dtypes'])
    from google.cloud import bigquery
    client = bigquery.Client(project=resolve_project_id())
    q = f"""
    SELECT number, timestamp AS block_timestamp, transaction_count, size, weight
    FROM `bigquery-public-data.crypto_bitcoin.blocks`
    WHERE timestamp >= TIMESTAMP('{YEAR_START}') AND timestamp < TIMESTAMP('{YEAR_END}')
    ORDER BY number
    """
    try:
        r = client.query(q).to_dataframe()
    except Exception as e:
        raise RuntimeError("BigQuery failed (project id / permission / API not enabled). Easiest fix: set DATA_SOURCE='api'. Original: " + str(e))
    r.to_parquet(CACHE); return r

fetch = fetch_from_api if DATA_SOURCE == "api" else fetch_from_bigquery

def coverage_problems(r):
    msgs = []; ts = pd.to_datetime(r["block_timestamp"], utc=True)
    if len(r) < MIN_BLOCKS: msgs.append(f"only {len(r)} blocks (< {MIN_BLOCKS})")
    if ts.min() > pd.Timestamp(YEAR_START, tz="UTC") + pd.Timedelta(days=1): msgs.append(f"starts late: {ts.min()}")
    if ts.max() < pd.Timestamp(YEAR_END, tz="UTC") - pd.Timedelta(days=3): msgs.append(f"ends early: {ts.max()}")
    nn = np.sort(r["number"].values)
    if int(r["number"].duplicated().sum()): msgs.append("duplicate block numbers")
    if int((np.diff(nn) != 1).sum()): msgs.append("gaps in block numbers")
    return msgs

used_cache = os.path.exists(CACHE)
raw = pd.read_parquet(CACHE) if used_cache else fetch()
if not QUICK:
    probs = coverage_problems(raw)
    if probs and used_cache:
        print("STALE/PARTIAL CACHE ->", probs, "\nRe-fetching ...")
        os.remove(CACHE); raw = fetch(); probs = coverage_problems(raw)
    assert not probs, f"Data coverage check FAILED: {probs}"
    print("Coverage guard PASSED:", len(raw), "blocks,", raw.block_timestamp.min(), "->", raw.block_timestamp.max())
else:
    raw = raw.iloc[:12000].copy(); print("QUICK mode: coverage guard skipped")
display(raw.head())

In [ ]:
# 3) FEATURES (leak-free) + PURGED CHRONOLOGICAL SPLIT + DATASET AUDIT
# Unit = one Bitcoin block.  Row i uses ONLY information available after block i-1.
TARGET_INT = "block_time"   # inter-block timestamp interval (miner-reported)  -> Task A
TARGET_DEM = "demand"       # mean tx_count of blocks i..i+5 (workload intensity, forward window) -> Task B

def build_base_features(raw):
    d = raw.sort_values("number").reset_index(drop=True).copy()
    d["block_timestamp"] = pd.to_datetime(d["block_timestamp"], utc=True)
    d["dt_raw"] = d["block_timestamp"].diff().dt.total_seconds()
    n_neg = int((d["dt_raw"] < 0).sum())
    d[TARGET_INT] = d["dt_raw"].clip(lower=0)
    tx, sz, wt, bt = d["transaction_count"], d["size"], d["weight"], d[TARGET_INT]
    d[TARGET_DEM] = tx.iloc[::-1].rolling(6).mean().iloc[::-1]          # forward window i..i+5
    d["prev_tx"], d["prev_size"], d["prev_weight"] = tx.shift(1), sz.shift(1), wt.shift(1)
    d["hour"]      = d["block_timestamp"].shift(1).dt.hour
    d["dayofweek"] = d["block_timestamp"].shift(1).dt.dayofweek
    d["tx_ma_5"]     = tx.shift(1).rolling(5).mean()
    d["tx_std_5"]    = tx.shift(1).rolling(5).std()
    d["size_ma_5"]   = sz.shift(1).rolling(5).mean()
    d["weight_ma_5"] = wt.shift(1).rolling(5).mean()
    d["tx_ma_36"]    = tx.shift(1).rolling(36).mean()
    d["tx_ma_144"]   = tx.shift(1).rolling(144, min_periods=36).mean()
    d["prev_block_time"]       = bt.shift(1)
    d["prev_block_time_ma_3"]  = bt.shift(1).rolling(3).mean()
    d["prev_block_time_std_3"] = bt.shift(1).rolling(3).std()
    d["tx_pct_change"]     = tx.pct_change().replace([np.inf, -np.inf], np.nan).shift(1)
    d["weight_pct_change"] = wt.pct_change().replace([np.inf, -np.inf], np.nan).shift(1)
    return d, n_neg

base, n_neg_dt = build_base_features(raw)
BASE_COLS = ["prev_tx","prev_size","prev_weight","hour","dayofweek","tx_ma_5","tx_std_5","size_ma_5","weight_ma_5",
             "tx_ma_36","tx_ma_144","prev_block_time","prev_block_time_ma_3","prev_block_time_std_3",
             "tx_pct_change","weight_pct_change"]
df = base.dropna(subset=BASE_COLS + [TARGET_INT, TARGET_DEM]).reset_index(drop=True)
ts_ = df["block_timestamp"]
if QUICK:
    n = len(df); i_tr = int(n * TRAIN_FRAC); i_va = int(n * (TRAIN_FRAC + VAL_FRAC))
else:                                              # DATE-BASED, out-of-time split
    i_tr = int((ts_ < pd.Timestamp(T_TRAIN_END, tz="UTC")).sum()); i_va = int((ts_ < pd.Timestamp(T_VAL_END, tz="UTC")).sum()); n = len(df)
    assert i_tr > 30000 and i_va - i_tr > 10000 and n - i_va > 10000, f"unexpected split sizes {i_tr}, {i_va - i_tr}, {n - i_va}"
a_tr, a_va = i_tr - EMBARGO, i_va - EMBARGO           # purged boundaries (train: [:a_tr], val: [i_tr:a_va], test: [i_va:])
tr = df.iloc[:a_tr]
NORM = dict(tx_mean=float(tr.prev_tx.mean()), tx_std=float(tr.prev_tx.std()),
            tx_max=float(tr.prev_tx.max()), w_max=float(tr.prev_weight.max()))
df["burst_flag"]     = (df["prev_tx"] > NORM["tx_mean"] + NORM["tx_std"]).astype(int)
df["queue_pressure"] = 0.5 * df["prev_tx"] / NORM["tx_max"] + 0.5 * df["prev_weight"] / NORM["w_max"]

FEATURES_INT = ["prev_tx","prev_size","prev_weight","hour","dayofweek","tx_ma_5","tx_std_5","size_ma_5","weight_ma_5",
                "prev_block_time","prev_block_time_ma_3","prev_block_time_std_3","burst_flag","queue_pressure",
                "tx_pct_change","weight_pct_change"]
FEATURES_DEM = FEATURES_INT + ["tx_ma_36", "tx_ma_144"]
SIM["DT_MEAN"] = float(df[TARGET_INT].iloc[:a_tr].mean())
DEMAND_SCALE = float(df[TARGET_DEM].iloc[:a_tr].mean())   # TRAINING-only reference for normalising demand/forecasts (no val/test statistic enters the controller)

audit = dict(
    protocol="train 2020-01..09 | validation 2020-10..12 (all tuning) | held-out test 2021-01..03 (re-used across design iterations; not independent)", config_hash=CONFIG_HASH,
    observation_unit="one Bitcoin block", code_version=CODE_VERSION, demand_scale_train_mean=DEMAND_SCALE, data_source=DATA_SOURCE, data_source_detail=("public Esplora-compatible block APIs (mempool.space primary, blockstream.info fallback)" if DATA_SOURCE == "api" else "Google BigQuery bigquery-public-data.crypto_bitcoin.blocks"), block_height_min=int(raw["number"].min()), block_height_max=int(raw["number"].max()), duplicate_heights=int(raw["number"].duplicated().sum()), date_range=[YEAR_START, YEAR_END],
    blocks_fetched=int(len(raw)), rows_after_feature_construction=int(n), rows_dropped=int(len(raw) - n),
    n_features_interval_task=len(FEATURES_INT), n_features_demand_task=len(FEATURES_DEM),
    embargo_blocks=EMBARGO, n_train=int(a_tr), n_val=int(a_va - i_tr), n_test=int(n - i_va),
    train_range=[str(df.block_timestamp.iloc[0]), str(df.block_timestamp.iloc[a_tr-1])],
    val_range=[str(df.block_timestamp.iloc[i_tr]), str(df.block_timestamp.iloc[a_va-1])],
    test_range=[str(df.block_timestamp.iloc[i_va]), str(df.block_timestamp.iloc[-1])],
    negative_intervals_clipped_to_0=n_neg_dt,
    interval_mean_s=float(df[TARGET_INT].mean()), interval_cv=float(df[TARGET_INT].std() / df[TARGET_INT].mean()),
    interval_lag1_autocorr_train=float(df[TARGET_INT].iloc[:a_tr].autocorr(1)),
    demand_lag1_autocorr_train=float(df[TARGET_DEM].iloc[:a_tr].autocorr(1)),
    demand_mean_train_val_test=[float(df[TARGET_DEM].iloc[:a_tr].mean()), float(df[TARGET_DEM].iloc[i_tr:a_va].mean()),
                                float(df[TARGET_DEM].iloc[i_va:].mean())],
    norm_constants_from_train=NORM,
)
json.dump(audit, open(f"{OUT_DIR}/dataset_audit.json", "w"), indent=2)
print(json.dumps(audit, indent=2))

In [ ]:
# 4) TWO PREDICTION TASKS (selection on VALIDATION, test touched once)
def make_models():
    ne = 100 if QUICK else 300
    wrap = lambda m: Pipeline([("sc", StandardScaler()), ("m", m)])
    return {
        "RandomForest": wrap(RandomForestRegressor(n_estimators=ne, max_depth=12, min_samples_split=5, random_state=SEED, n_jobs=-1)),
        "GradientBoosting": wrap(GradientBoostingRegressor(n_estimators=ne, learning_rate=0.03, max_depth=3, random_state=SEED)),
        "MLP": wrap(MLPRegressor(hidden_layer_sizes=(128, 64), activation="relu", solver="adam", learning_rate_init=0.0008,
                                 max_iter=700, early_stopping=True, random_state=SEED)),
        "Ridge": wrap(Ridge(alpha=1.0)),
    }
SL = dict(tr=slice(0, a_tr), va=slice(i_tr, a_va), te=slice(i_va, n))

def metrics(yt, yp):
    return dict(MAE=mean_absolute_error(yt, yp), RMSE=float(np.sqrt(mean_squared_error(yt, yp))), R2=r2_score(yt, yp))

def run_task(tag, target, feats, baselines, model_names):
    yy, Xf = df[target], df[feats]
    PV, PT, fitted, rows = {}, {}, {}, []
    for bn, fn in baselines.items():
        PV[bn], PT[bn] = fn(Xf.iloc[SL["va"]]), fn(Xf.iloc[SL["te"]])
    for nm in model_names:
        m = make_models()[nm]; t0 = time.perf_counter()
        m.fit(Xf.iloc[SL["tr"]], yy.iloc[SL["tr"]]); fitted[nm] = m
        PV[nm], PT[nm] = m.predict(Xf.iloc[SL["va"]]), m.predict(Xf.iloc[SL["te"]])
        print(f"[{tag}] {nm}: trained in {time.perf_counter() - t0:.1f}s")
    for nm in PV:
        mv, mt = metrics(yy.iloc[SL["va"]], PV[nm]), metrics(yy.iloc[SL["te"]], PT[nm])
        rows.append(dict(Model=nm, **{f"val_{k}": v for k, v in mv.items()}, **{f"test_{k}": v for k, v in mt.items()}))
    table = pd.DataFrame(rows).set_index("Model")
    sel = table.loc[model_names].sort_values("val_RMSE").index[0]
    best_base = table.loc[list(baselines)].sort_values("val_RMSE").index[0]
    return dict(tag=tag, target=target, feats=feats, table=table, SEL=sel, best_base=best_base, fitted=fitted,
                PV=PV, PT=PT, ref_mae=float(table.loc[sel, "val_MAE"]),
                y_va=yy.iloc[SL["va"]].values, y_te=yy.iloc[SL["te"]].values)

seas = df.iloc[:a_tr].groupby(df["hour"].iloc[:a_tr].astype(int))[TARGET_DEM].mean()
BASE_INT = {"Baseline: train mean": lambda X: np.full(len(X), df[TARGET_INT].iloc[:a_tr].mean()),
            "Baseline: previous interval": lambda X: X["prev_block_time"].values}
BASE_DEM = {"Baseline: train mean": lambda X: np.full(len(X), df[TARGET_DEM].iloc[:a_tr].mean()),
            "Baseline: persistence (5-block mean)": lambda X: X["tx_ma_5"].values,
            "Baseline: 36-block mean": lambda X: X["tx_ma_36"].values,
            "Baseline: hour-of-day mean": lambda X: X["hour"].astype(int).map(seas).fillna(seas.mean()).values}

TA = run_task("A: interval", TARGET_INT, FEATURES_INT, BASE_INT, ["RandomForest", "GradientBoosting", "MLP"])
TB = run_task("B: demand",   TARGET_DEM, FEATURES_DEM, BASE_DEM, ["Ridge", "RandomForest", "GradientBoosting", "MLP"])
for T in (TA, TB):
    print(f"\n===== Task {T['tag']}  (target = {T['target']}) =====")
    display(T["table"].round(4)); T["table"].to_csv(f"{OUT_DIR}/table_prediction_{T['tag'][0]}.csv")
    print("Selected on validation:", T["SEL"], "| best baseline on validation:", T["best_base"])

In [ ]:
# 5) PREDICTION STATISTICS THAT RESPECT TEMPORAL DEPENDENCE
def block_bootstrap_ci(a, b, block=144, n_boot=3000, seed=SEED):
    rng = np.random.default_rng(seed); a, b = np.asarray(a, float), np.asarray(b, float)
    N = len(a); k = int(np.ceil(N / block)); out = np.empty(n_boot)
    for i in range(n_boot):
        starts = rng.integers(0, N - block + 1, k)
        idx = (starts[:, None] + np.arange(block)[None, :]).ravel()[:N]
        out[i] = 100 * (a[idx].mean() - b[idx].mean()) / a[idx].mean()
    return np.percentile(out, [2.5, 97.5])

def ljung_box(res, lags=(10, 50)):
    res = np.asarray(res, float) - np.mean(res); N = len(res)
    acf = np.array([np.sum(res[k:] * res[:N - k]) / np.sum(res**2) for k in range(1, max(lags) + 1)])
    return {L: dict(Q=float(N*(N+2)*np.sum(acf[:L]**2/(N-np.arange(1, L+1)))),
                    p=float(1 - sps.chi2.cdf(N*(N+2)*np.sum(acf[:L]**2/(N-np.arange(1, L+1))), L))) for L in lags}

PRED_STATS = {}
for T in (TA, TB):
    e_sel = np.abs(T["y_te"] - T["PT"][T["SEL"]]); rows = []
    for bn in [b for b in T["PT"] if b.startswith("Baseline")]:
        e_b = np.abs(T["y_te"] - T["PT"][bn]); lo, hi = block_bootstrap_ci(e_b, e_sel)
        rows.append(dict(task=T["tag"], model=T["SEL"], vs=bn, MAE_improvement_pct=100*(e_b.mean()-e_sel.mean())/e_b.mean(),
                         CI95_low=lo, CI95_high=hi, significant=bool(lo > 0)))
    PRED_STATS[T["tag"]] = pd.DataFrame(rows)
    print(f"\n--- Task {T['tag']}: {T['SEL']} vs baselines (moving-block bootstrap, block=144) ---")
    display(PRED_STATS[T["tag"]].round(3))
    print("Ljung-Box on test residuals:", json.dumps(ljung_box(T["y_te"] - T["PT"][T["SEL"]])))
pd.concat(PRED_STATS.values()).to_csv(f"{OUT_DIR}/table_prediction_block_bootstrap.csv", index=False)

In [ ]:
# 6) CLOSED LOOP on the demand forecaster: walk-forward + Page-Hinkley + recalibration (real 2020 test period)
class PageHinkley:
    def __init__(self, delta=0.05, lam=1.0): self.delta, self.lam = delta, lam; self.reset()
    def reset(self): self.n, self.mean, self.cum, self.min = 0, 0.0, 0.0, 0.0
    def update(self, x):
        self.n += 1; self.mean += (x - self.mean) / self.n
        self.cum += x - self.mean - self.delta; self.min = min(self.min, self.cum)
        return (self.cum - self.min) > self.lam

def fit_initial(T, dfx=None):
    dfx = df if dfx is None else dfx; lo0 = max(0, a_va - WINDOW); pipe = make_models()[T["SEL"]]
    pipe.fit(dfx[T["feats"]].iloc[lo0:a_va], dfx[T["target"]].iloc[lo0:a_va]); return pipe     # train+val (purged)

def walk_forward(T, mode, init_pipe, dfx=None):
    """mode: 'frozen' (no feedback) | 'periodic' | 'drift' (feedback-triggered recalibration on the latest RECAL_WINDOW blocks)."""
    dfx = df if dfx is None else dfx; Xf, yy = dfx[T["feats"]], dfx[T["target"]]
    pipe = copy.deepcopy(init_pipe); ph = PageHinkley(PH_DELTA, PH_LAMBDA); n_te = n - i_va
    preds, chunk_mae, events, rt, last = np.empty(n_te), [], [], 0.0, -10**9
    for k, s in enumerate(range(0, n_te, CHUNK)):
        a, b = i_va + s, min(i_va + s + CHUNK, n)
        p = pipe.predict(Xf.iloc[a:b]); preds[s:b - i_va] = p
        m = mean_absolute_error(yy.iloc[a:b], p); chunk_mae.append(m)
        if mode == "drift":      do = ph.update(m / T["ref_mae"]) and (k - last >= COOLDOWN)
        elif mode == "periodic": do = ((k + 1) % PERIODIC_EVERY == 0)
        else:                    do = False
        if do and b < n:                                                   # only labels fully observed (<= b-EMBARGO) are legal
            t0 = time.perf_counter(); pipe = make_models()[T["SEL"]]; l_ = max(0, b - EMBARGO - RECAL_WINDOW)
            pipe.fit(Xf.iloc[l_:b - EMBARGO], yy.iloc[l_:b - EMBARGO]); rt += time.perf_counter() - t0
            events.append(k); ph.reset(); last = k
    return dict(preds=preds, chunk_mae=np.array(chunk_mae), events=events, retrain_time=rt)

INIT_B = fit_initial(TB)
WF = {m: walk_forward(TB, m, INIT_B) for m in ["frozen", "periodic", "drift"]}
wf_table = pd.DataFrame([dict(mode=m, **metrics(TB["y_te"], r["preds"]), n_recalibrations=len(r["events"]),
                              retrain_seconds=r["retrain_time"]) for m, r in WF.items()])
display(wf_table.round(4)); wf_table.to_csv(f"{OUT_DIR}/table_closed_loop_modes.csv", index=False)
e_fro = np.abs(TB["y_te"] - WF["frozen"]["preds"])
for m in ["periodic", "drift"]:
    e_m = np.abs(TB["y_te"] - WF[m]["preds"]); lo, hi = block_bootstrap_ci(e_fro, e_m)
    print(f"{m} vs frozen: MAE change {100*(e_fro.mean()-e_m.mean())/e_fro.mean():+.2f}%  CI95 [{lo:+.2f}, {hi:+.2f}]  (positive = recalibration helps)")
fig, ax = plt.subplots(figsize=(11, 3.8))
for m, r in WF.items(): ax.plot(r["chunk_mae"], label=m, lw=1.2)
for k in WF["drift"]["events"]: ax.axvline(k, color="k", ls=":", alpha=.4)
ax.set_xlabel(f"cycle ({CHUNK} blocks)"); ax.set_ylabel("chunk MAE (tx/block)"); ax.set_title("Demand-forecast error, walk-forward, REAL held-out test period (2021-Q1) (dotted = recalibration)")
ax.legend(); plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_closed_loop_mae.png", dpi=200); plt.show()
fig, ax = plt.subplots(figsize=(11, 3.8)); m_ = slice(0, min(1500, len(TB["y_te"])))
ax.plot(TB["y_te"][m_], label="actual demand", lw=.8); ax.plot(WF["drift"]["preds"][m_], label="forecast (closed loop)", lw=.8)
ax.set_xlabel("test block"); ax.set_ylabel("tx / block"); ax.legend(); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig_demand_forecast.png", dpi=200); plt.show()
PREDS_CTRL = WF["drift"]["preds"]

In [ ]:
# 6b) DRIFT-INJECTION STRESS TEST: does the closed loop detect a regime change and recover?
# A real test period may contain little or no drift, in which case the closed loop cannot show a benefit there.  We therefore inject a KNOWN,
# SYNTHETIC drift (clearly labelled): from the middle of the test period every block carries DRIFT_SCALE x more transactions/bytes/weight.
i_drift = i_va + (n - i_va) // 2; drift_cycle = (i_drift - i_va) // CHUNK
raw_mod = raw.sort_values("number").reset_index(drop=True).copy()
mask = raw_mod["number"] >= df["number"].iloc[i_drift]
for c_ in ["transaction_count", "size", "weight"]:
    raw_mod[c_] = raw_mod[c_].astype(float); raw_mod.loc[mask, c_] = raw_mod.loc[mask, c_] * DRIFT_SCALE
base_m, _ = build_base_features(raw_mod)
dfx = base_m.dropna(subset=BASE_COLS + [TARGET_INT, TARGET_DEM]).reset_index(drop=True)
assert len(dfx) == n and (dfx["number"].values == df["number"].values).all()
dfx["burst_flag"] = (dfx["prev_tx"] > NORM["tx_mean"] + NORM["tx_std"]).astype(int)
dfx["queue_pressure"] = 0.5 * dfx["prev_tx"] / NORM["tx_max"] + 0.5 * dfx["prev_weight"] / NORM["w_max"]
y_stress = dfx[TARGET_DEM].iloc[i_va:].values
WFS = {m: walk_forward(TB, m, INIT_B, dfx) for m in ["frozen", "drift"]}
rows = []
for m, r in WFS.items():
    post = slice(max(0, i_drift - i_va), None)
    rows.append(dict(mode=m, MAE_all=mean_absolute_error(y_stress, r["preds"]), MAE_after_drift=mean_absolute_error(y_stress[post], r["preds"][post]),
                     n_recalibrations=len(r["events"]), first_detection_cycle=(min([e for e in r["events"] if e >= drift_cycle], default=None)),
                     drift_starts_cycle=drift_cycle))
stress = pd.DataFrame(rows); display(stress.round(2)); stress.to_csv(f"{OUT_DIR}/table_closed_loop_stress.csv", index=False)
pf = slice(max(0, i_drift - i_va), None)
ea, eb = np.abs(y_stress[pf] - WFS["frozen"]["preds"][pf]), np.abs(y_stress[pf] - WFS["drift"]["preds"][pf])
lo_s, hi_s = block_bootstrap_ci(ea, eb)
STRESS_CI = (100 * (ea.mean() - eb.mean()) / ea.mean(), lo_s, hi_s)
print(f"After the injected drift: recalibration changes MAE by {STRESS_CI[0]:+.1f}% CI95 [{lo_s:+.1f}, {hi_s:+.1f}] (positive = better than frozen)")
fig, ax = plt.subplots(figsize=(11, 3.8))
for m, r in WFS.items(): ax.plot(r["chunk_mae"], label=m, lw=1.2)
ax.axvline(drift_cycle, color="r", ls="--", label="injected drift"); [ax.axvline(k, color="k", ls=":", alpha=.4) for k in WFS["drift"]["events"]]
ax.set_xlabel(f"cycle ({CHUNK} blocks)"); ax.set_ylabel("chunk MAE"); ax.set_title("Closed-loop stress test (SYNTHETIC injected workload drift)"); ax.legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_closed_loop_stress.png", dpi=200); plt.show()

In [ ]:
# 7) TRACE-DRIVEN QUEUE SIMULATOR
# World (identical for every policy):
#  * block times      = REAL inter-block intervals of the period
#  * arrival process  = Poisson with intensity proportional to the REAL demand (forward 6-block mean tx_count),
#                       scaled so the average offered load = rho x block capacity (operator-calibrated mean rate)
#  * tx attributes    = class (HIGH/NORMAL/LOW), size (lognormal), off-loadable flag; PRE-GENERATED per (rho, seed)
# Decision timing: the controller chooses its regime at the START of each interval, seeing only the backlog left after the
# previous block + (for ADAPTIVE) a forecast of the demand of the coming interval.  ORACLE sees the true arrival count.
REG = {
    "STATIC":   dict(order="fifo", elig=(),     B=1),    # executed baseline: FIFO, full payload on-chain
    "FAST":     dict(order="fifo", elig=(),     B=1),
    "FIFO_OFF": dict(order="fifo", elig=(1, 2), B=10),   # static hybrid storage, no priority
    "PRIO":     dict(order="prio", elig=(),     B=1),    # priority+aging only
    "MID":      dict(order="prio", elig=(2,),   B=10),
    "CONG":     dict(order="prio", elig=(1, 2), B=25),
}
FIXED_CANDIDATES = ["FIFO_OFF", "PRIO", "MID", "CONG"]

def make_world(sl, rel_hat_map):
    dt = df[TARGET_INT].iloc[sl].values.astype(float); dem = df[TARGET_DEM].iloc[sl].values.astype(float); sc = DEMAND_SCALE   # fixed TRAINING mean: no information from the evaluated period
    rh = {k: np.clip(np.asarray(v) / sc, 0.2, 3.0) for k, v in rel_hat_map.items()}
    rh["ADAPTIVE_CONST"] = np.ones(len(dt))          # ablation: knows only the AVERAGE load level, no time-varying forecast
    return dict(dt=dt, rel=dem / sc, rel_hat=rh)

def make_arrivals(world, rho, seed, S):
    rng = np.random.default_rng(seed); dt = world["dt"]
    n_arr = rng.poisson(rho * S["CAP"] / 600.0 * world["rel"] * dt); N = int(n_arr.sum())
    t_end = np.cumsum(dt); t_start = t_end - dt; blk = np.repeat(np.arange(len(dt)), n_arr)
    times = np.sort(t_start[blk] + rng.random(N) * dt[blk]); sg = S["SIZE_SIGMA"]
    return dict(t=times, cls=rng.choice(3, N, p=S["CLASS_P"]), size=rng.lognormal(-sg**2 / 2, sg, N),
                off=rng.random(N) < S["OFFLOADABLE_P"], offs=np.concatenate([[0], np.cumsum(n_arr)]), t_end=t_end)

def select_block(t, q_t, q_c, q_s, q_o, reg, S):
    nq = len(q_t)
    if nq == 0: return np.zeros(0, bool), 0.0
    if reg["order"] == "fifo": o = np.arange(nq)
    else: o = np.argsort(-(S["CLASS_W"][q_c] + (t - q_t) / S["T_AGE"]), kind="stable")
    elig = (np.isin(q_c, reg["elig"]) & q_o) if reg["elig"] else np.zeros(nq, bool)
    el_o, pos, so, to = elig[o], np.arange(nq), q_s[o], q_t[o]
    ne_pos, ne_cost = pos[~el_o], so[~el_o]; el_pos = pos[el_o]; m = len(el_pos); B = reg["B"]
    n_full = m // B; rem = m - n_full * B; nb = n_full
    if rem > 0 and (t - to[el_pos[n_full * B:]]).max() > S["MAX_WAIT_PARTIAL"]: nb += 1
    first = el_pos[::B][:nb] if nb > 0 else np.zeros(0, int)
    ranks = np.concatenate([ne_pos, first]); costs = np.concatenate([ne_cost, np.full(nb, S["ANCHOR"])])
    oi = np.argsort(ranks, kind="stable"); cum = np.cumsum(costs[oi]); k = int(np.searchsorted(cum, S["CAP"], side="right"))
    if k == 0: return np.zeros(nq, bool), 0.0
    inc = oi[:k]; is_ne = inc < len(ne_pos); inc_o = np.zeros(nq, bool); inc_o[ne_pos[inc[is_ne]]] = True
    if nb > 0:
        inc_b = np.zeros(nb, bool); inc_b[inc[~is_ne] - len(ne_pos)] = True
        bid = np.arange(m) // B; ok = bid < nb; inc_o[el_pos[ok][inc_b[bid[ok]]]] = True
    mask = np.zeros(nq, bool); mask[o[inc_o]] = True
    return mask, float(cum[k - 1])

def decide(policy, j, br0, arr, world, rho, S, P):
    """Regime chosen at the START of interval j (priority ordering is ALWAYS on; only the off-chain batching level adapts):
       s <= lo -> PRIO (no batching) | lo < s < hi -> MID | s >= hi -> CONG.
       FEEDBACK: s = backlog (in block capacities).  ADAPTIVE: backlog + forecast load.  ORACLE: backlog + TRUE future arrivals.
       Statistic mode 'I' = coming interval (s = backlog + load);  'H' = predicted backlog growth over the next H blocks."""
    if policy == "STATIC":     return "STATIC"
    if policy == "PRIO_ONLY":  return "PRIO"
    if policy == "BEST_FIXED": return P["BEST_FIXED"]
    if policy.startswith("FIXED:"): return policy.split(":")[1]
    H = S["H"]
    if policy == "FEEDBACK":
        s, (lo, hi) = br0, P["FEEDBACK"]
    else:
        key = policy if policy in ("ORACLE", "ADAPTIVE_CONST") else "ADAPTIVE"; lo, hi, mode = P[key]
        if policy == "ORACLE":
            nb_ = len(arr["offs"]) - 1
            if mode == "I": s = br0 + (arr["offs"][j + 1] - arr["offs"][j]) / S["CAP"]
            else: je = min(j + H, nb_); s = max(0.0, br0 + (arr["offs"][je] - arr["offs"][j]) / S["CAP"] - (je - j))
        else:
            u_hat = rho * world["rel_hat"][policy][j] * S["DT_MEAN"] / 600.0
            s = br0 + u_hat if mode == "I" else max(0.0, br0 + H * (u_hat - 1.0))
    return "CONG" if s >= hi else ("PRIO" if s <= lo else "MID")

def simulate(arr, world, policy, rho, S, P):
    nb = len(world["dt"]); offs = arr["offs"]
    q_t, q_c, q_s, q_o = np.empty(0), np.empty(0, int), np.empty(0), np.empty(0, bool)
    L, C, A, CT, used, backlog, counts = [], [], [], [], np.zeros(nb), np.zeros(nb), {}
    for j in range(nb):
        rname = decide(policy, j, len(q_t) / S["CAP"], arr, world, rho, S, P); counts[rname] = counts.get(rname, 0) + 1
        s, e = offs[j], offs[j + 1]
        if e > s:
            q_t = np.concatenate([q_t, arr["t"][s:e]]); q_c = np.concatenate([q_c, arr["cls"][s:e]])
            q_s = np.concatenate([q_s, arr["size"][s:e]]); q_o = np.concatenate([q_o, arr["off"][s:e]])
        t = arr["t_end"][j]; mask, u = select_block(t, q_t, q_c, q_s, q_o, REG[rname], S)
        if mask.any():
            L.append(t - q_t[mask]); C.append(q_c[mask]); A.append(q_t[mask]); CT.append(np.full(int(mask.sum()), t)); keep = ~mask
            q_t, q_c, q_s, q_o = q_t[keep], q_c[keep], q_s[keep], q_o[keep]
        used[j] = u; backlog[j] = len(q_t)
    cat = lambda l, d: np.concatenate(l) if l else np.empty(0, d)
    return dict(lat=cat(L, float), cls=cat(C, int), arr_t=cat(A, float), comp_t=cat(CT, float), arrivals_t=arr["t"], arrivals_cls=arr["cls"], used=used, backlog=backlog, q_t=q_t, q_c=q_c,
                t_end=arr["t_end"][-1], t_warm=arr["t_end"][int(S["WARMUP_FRAC"] * nb)], j_warm=int(S["WARMUP_FRAC"] * nb),
                n_arrived=len(arr["t"]), reg=counts)

def summarize(res, S, seg=0):
    """seg=0 -> whole period after warm-up; seg=1..4 -> consecutive quarters of it.
    Percentiles INCLUDE still-unfinished tx (their current wait = lower bound) so starvation cannot hide."""
    edges = np.linspace(res["t_warm"], res["t_end"], 5)
    lo, hi = (res["t_warm"], np.inf) if seg == 0 else (edges[seg - 1], edges[seg] if seg < 4 else np.inf)
    m = (res["arr_t"] >= lo) & (res["arr_t"] < hi); lat, cl = res["lat"][m], res["cls"][m]
    um = (res["q_t"] >= lo) & (res["q_t"] < hi); ua, uc = res["t_end"] - res["q_t"][um], res["q_c"][um]
    rows = []
    for name, sel in [("ALL", None), ("HIGH", 0), ("NORMAL", 1), ("LOW", 2)]:
        l = lat if sel is None else lat[cl == sel]; a = ua if sel is None else ua[uc == sel]; both = np.concatenate([l, a])
        pc = np.percentile(both, [50, 95, 99]) if len(both) else [np.nan] * 3
        rows.append(dict(cls=name, n_done=len(l), n_unfinished=len(a), mean=both.mean() if len(both) else np.nan,
                         std=both.std() if len(both) else np.nan, p50=pc[0], p95=pc[1], p99=pc[2],
                         max_wait=both.max() if len(both) else np.nan, sla_hit=(l <= S["SLA"]).sum() / max(len(both), 1)))
    bl = res["backlog"][res["j_warm"]:]; tot = sum(res["reg"].values())
    t_hi = res["t_end"] if not np.isfinite(hi) else hi; dur = max(t_hi - lo, 1.0)           # window [lo, t_hi)
    done_w = (res["comp_t"] >= lo) & (res["comp_t"] < t_hi) if np.isfinite(hi) else (res["comp_t"] >= lo)
    arr_w = (res["arrivals_t"] >= lo) & (res["arrivals_t"] < t_hi) if np.isfinite(hi) else (res["arrivals_t"] >= lo)
    comp_cls = res["cls"][done_w]; arr_cls = res["arrivals_cls"][arr_w]
    glob = dict(completed_tps=done_w.sum() / dur,                 # confirmations per second INSIDE the window (true completed throughput)
                arrival_tps=arr_w.sum() / dur,                    # offered arrival rate in the same window
                completed_to_arrived_ratio=done_w.sum() / max(arr_w.sum(), 1),
                completed_tps_HIGH=(comp_cls == 0).sum() / dur, completed_tps_NORMAL=(comp_cls == 1).sum() / dur, completed_tps_LOW=(comp_cls == 2).sum() / dur,
                arrival_tps_HIGH=(arr_cls == 0).sum() / dur, arrival_tps_NORMAL=(arr_cls == 1).sum() / dur, arrival_tps_LOW=(arr_cls == 2).sum() / dur,
                onchain_units_per_tx=res["used"].sum() / max(len(res["lat"]), 1), backlog_mean=bl.mean(),
                backlog_p95=np.percentile(bl, 95), backlog_max=bl.max(), backlog_final=res["backlog"][-1],
                frac_time_overloaded=float((bl > 5 * S["CAP"]).mean()),
                **{f"share_{k}": res["reg"].get(k, 0) / tot for k in ["STATIC", "FAST", "FIFO_OFF", "PRIO", "MID", "CONG"]})
    return rows, glob
print("Simulator ready.")

In [ ]:
# 8) TUNE THE CONTROLLERS ON THE VALIDATION PERIOD ONLY (no test-set tuning; every policy gets the same tuning budget)
# Pre-specified two-stage objective (epsilon-constraint):
#   latency objective  L = p95(HIGH) + mean-class p95   (+ 1e5 x violation of "every class SLA hit-rate >= SLA_FLOOR")
#   stage 1: L* = best latency objective reachable by ANY non-oracle policy/configuration
#   stage 2: each policy picks, among its configurations with L <= (1+EPS_TOL) x L*, the one with the LEAST off-chain use.
# => adaptation is rewarded for keeping payloads ON-CHAIN whenever the service level allows it.
dem_val_pred = TB["fitted"][TB["SEL"]].predict(df[FEATURES_DEM].iloc[SL["va"]])
W_VAL = make_world(SL["va"], {"ADAPTIVE": dem_val_pred})
ARR_VAL = {(rho, s_): make_arrivals(W_VAL, rho, 1000 + 17 * i + s_, SIM) for i, rho in enumerate(TUNE_RHOS) for s_ in range(TUNE_SEEDS)}

def objective_from_rows(rows):
    p = {r["cls"]: r for r in rows}; sla_min = min(p[c]["sla_hit"] for c in CLS)
    return p["HIGH"]["p95"] + np.mean([p[c]["p95"] for c in CLS]) + 1e5 * max(0.0, SLA_FLOOR - sla_min)

def tune_eval(policy, P):
    objs, offl = [], []
    for (rho, _), arr in ARR_VAL.items():
        rows, g = summarize(simulate(arr, W_VAL, policy, rho, SIM, P), SIM); objs.append(objective_from_rows(rows)); offl.append(1 - g["onchain_units_per_tx"])
    return float(np.mean(objs)), float(np.mean(offl))

grid = [(lo, hi) for lo in GRID_LO for hi in GRID_HI if hi > lo]
cands = []
st_obj, st_off = tune_eval("STATIC", {}); cands.append(dict(key="STATIC", params="-", obj=st_obj, offload=st_off))
for r in FIXED_CANDIDATES:
    o, f = tune_eval(f"FIXED:{r}", {}); cands.append(dict(key="BEST_FIXED", params=r, obj=o, offload=f))
for g in grid:
    o, f = tune_eval("FEEDBACK", {"FEEDBACK": g}); cands.append(dict(key="FEEDBACK", params=g, obj=o, offload=f))
for pol in ["ADAPTIVE", "ADAPTIVE_CONST", "ORACLE"]:
    for mode in ["I", "H"]:                      # statistic mode is a validation-tuned hyper-parameter
        for g in grid:
            p3 = (g[0], g[1], mode); o, f = tune_eval(pol, {pol: p3}); cands.append(dict(key=pol, params=p3, obj=o, offload=f))
tune_df = pd.DataFrame(cands); tune_df["params"] = tune_df["params"].astype(str); tune_df.to_csv(f"{OUT_DIR}/table_tuning_validation.csv", index=False)
L_star = tune_df[tune_df.key.isin(["BEST_FIXED", "FEEDBACK", "ADAPTIVE", "ADAPTIVE_CONST"])].obj.min()
TUNED = {}
for key in ["BEST_FIXED", "FEEDBACK", "ADAPTIVE", "ADAPTIVE_CONST", "ORACLE"]:
    pool = [c for c in cands if c["key"] == key]; ok = [c for c in pool if c["obj"] <= (1 + EPS_TOL) * L_star]
    best = min(ok, key=lambda c: c["offload"]) if ok else min(pool, key=lambda c: c["obj"])
    TUNED[key] = best["params"]; print(f" {key:15s} -> {best['params']!s:12s} latency-obj={best['obj']:9.0f}  off-chain share={best['offload']:.3f}  ({'within eps' if ok else 'NO config within eps: using best latency'})")
print(f"\nBest reachable latency objective L* = {L_star:.0f}; STATIC objective = {st_obj:,.0f} (>1e4 means SLA violated on validation)")

In [ ]:
# 9) SIMULATOR SANITY TESTS  (a failing test means: do NOT trust any simulator number)
rng_ = np.random.default_rng(7)
W_TEST = make_world(SL["te"], {"ADAPTIVE": WF["drift"]["preds"], "ADAPTIVE_FROZEN": WF["frozen"]["preds"],
                               "ADAPTIVE_SHUFFLED": rng_.permutation(WF["drift"]["preds"])})
checks = {}
arr0 = make_arrivals(W_TEST, 0.85, 123, SIM); arr0b = make_arrivals(W_TEST, 0.85, 123, SIM)
checks["identical arrivals for identical seed"] = bool(np.array_equal(arr0["t"], arr0b["t"]) and np.array_equal(arr0["size"], arr0b["size"]))
r1 = simulate(arr0, W_TEST, "ADAPTIVE", 0.85, SIM, TUNED); r2 = simulate(arr0, W_TEST, "ADAPTIVE", 0.85, SIM, TUNED)
checks["determinism (same input -> same output)"] = bool(np.array_equal(r1["lat"], r2["lat"]))
ok_cons, ok_cap = True, True
for pol in ["STATIC", "PRIO_ONLY", "BEST_FIXED", "FEEDBACK", "ADAPTIVE_CONST", "ADAPTIVE", "ORACLE"]:
    r = simulate(arr0, W_TEST, pol, 0.85, SIM, TUNED)
    ok_cons &= (r["n_arrived"] == len(r["lat"]) + len(r["q_t"]))          # no tx lost or duplicated
    ok_cap  &= bool(r["used"].max() <= SIM["CAP"] + 1e-9)                  # capacity never exceeded
checks["conservation: arrived = confirmed + still queued (all policies)"] = bool(ok_cons)
checks["block capacity never exceeded (all policies)"] = bool(ok_cap)
# no-harm test at an UNSEEN low load (0.2 < every tuning load): an adaptive controller must not be worse than static FIFO
arr_lo = make_arrivals(W_TEST, 0.2, 321, SIM); lo_res = {}
for pol in ["STATIC", "ADAPTIVE"]:
    rows, _ = summarize(simulate(arr_lo, W_TEST, pol, 0.2, SIM, TUNED), SIM); lo_res[pol] = {r["cls"]: r for r in rows}
worst = max((lo_res["ADAPTIVE"][c]["mean"] - lo_res["STATIC"][c]["mean"]) / lo_res["STATIC"][c]["mean"] for c in ["ALL"] + CLS)
checks[f"no-harm at unseen rho=0.2 (worst class mean latency change {100*worst:+.1f}%, tolerance +10%)"] = bool(worst <= 0.10)
sanity = pd.DataFrame({"test": list(checks), "passed": list(checks.values())}); display(sanity)
sanity.to_csv(f"{OUT_DIR}/table_sanity_tests.csv", index=False)
if not sanity.passed.all(): print("!! A SANITY TEST FAILED - investigate before using any result.")

In [ ]:
# 10) MAIN EXPERIMENT (identical arrivals across policies; 3 loads x seeds; whole period + 4 consecutive segments)
POLICIES = ["STATIC", "PRIO_ONLY", "BEST_FIXED", "FEEDBACK", "ADAPTIVE_CONST", "ADAPTIVE_SHUFFLED", "ADAPTIVE_FROZEN", "ADAPTIVE", "ORACLE"]
all_rows, keep, t0 = [], {}, time.perf_counter(); rho_mid = RHOS[len(RHOS) // 2]
for rho in RHOS:
    for sd in range(N_SEEDS):
        arr = make_arrivals(W_TEST, rho, SEED + sd, SIM)
        for pol in POLICIES:
            res = simulate(arr, W_TEST, pol, rho, SIM, TUNED)
            for seg in range(5):
                rws, glob = summarize(res, SIM, seg)
                for r in rws: all_rows.append(dict(rho=rho, seed=sd, policy=pol, seg=seg, **r, **glob))
            if sd == 0 and rho == rho_mid and pol in ("STATIC", "BEST_FIXED", "FEEDBACK", "ADAPTIVE"): keep[pol] = res
    print(f"rho={rho} done ({time.perf_counter() - t0:.0f}s)")
sim_df = pd.DataFrame(all_rows)
_tp = ["completed_tps", "completed_tps_HIGH", "completed_tps_NORMAL", "completed_tps_LOW", "arrival_tps", "completed_to_arrived_ratio"]
sim_df[sim_df.cls == "ALL"].query("seg == 0").groupby(["rho", "policy"])[_tp].agg(["mean", "std"]).to_csv(f"{OUT_DIR}/table_completed_tps.csv")
sim_df.to_csv(f"{OUT_DIR}/sim_raw_results.csv", index=False)

pd.set_option("display.width", 220)
for rho in RHOS:
    d = sim_df[(sim_df.rho == rho) & (sim_df.seg == 0)]
    print(f"\n================ offered load rho = {rho}  (mean over {N_SEEDS} seeds; latency in s, unfinished tx included) ================")
    tab = d.groupby(["cls", "policy"])[["mean", "std", "p95", "p99", "max_wait", "sla_hit"]].mean().unstack("policy")
    for col in ["mean", "p95", "p99", "sla_hit"]:
        print(f"-- {col} --"); display(tab[col].reindex(["ALL"] + CLS)[POLICIES].round(3))
    gl = d[d.cls == "ALL"].groupby("policy")[["completed_tps", "completed_tps_HIGH", "completed_tps_NORMAL", "completed_tps_LOW", "arrival_tps", "completed_to_arrived_ratio", "onchain_units_per_tx", "backlog_mean", "backlog_p95", "backlog_max",
                                              "backlog_final", "n_unfinished", "frac_time_overloaded", "share_PRIO", "share_MID", "share_CONG"]].mean().loc[POLICIES]
    print("-- COMPLETED throughput (tx confirmed per second in the window) / footprint / backlog / unfinished / regime usage --"); display(gl.round(4))

In [ ]:
# 11) PRE-SPECIFIED COMPARISONS (paired over seeds) + consistency over time
sim_df["macro_p95_helper"] = sim_df["p95"]
macro = (sim_df[sim_df.cls.isin(CLS)].groupby(["rho", "seed", "policy", "seg"])["p95"].mean().rename("macro_p95").reset_index())
allm = sim_df[sim_df.cls == "ALL"][["rho", "seed", "policy", "seg", "mean", "p95", "std"]].rename(columns={"mean": "all_mean", "p95": "all_p95", "std": "all_std"})
M = macro.merge(allm, on=["rho", "seed", "policy", "seg"])
M = M.merge(sim_df[(sim_df.cls == "HIGH")][["rho","seed","policy","seg","p95"]].rename(columns={"p95":"high_p95"}), on=["rho","seed","policy","seg"])
M = M.merge(sim_df[(sim_df.cls == "LOW")][["rho","seed","policy","seg","p95"]].rename(columns={"p95":"low_p95"}), on=["rho","seed","policy","seg"])
M = M.merge(sim_df[sim_df.cls == "ALL"][["rho","seed","policy","seg","onchain_units_per_tx","frac_time_overloaded"]].assign(offload=lambda d_: 1 - d_.onchain_units_per_tx)[["rho","seed","policy","seg","offload","frac_time_overloaded"]], on=["rho","seed","policy","seg"])
M = M.merge(sim_df[sim_df.cls.isin(CLS)].groupby(["rho","seed","policy","seg"])["sla_hit"].min().rename("sla_min").reset_index(), on=["rho","seed","policy","seg"])

def compare(A, B, metric, rho):
    """improvement of A over B in % (positive = A has lower latency), paired by seed (whole period) + segment win-rate"""
    a = M[(M.rho == rho) & (M.policy == A) & (M.seg == 0)].sort_values("seed")[metric].values
    b = M[(M.rho == rho) & (M.policy == B) & (M.seg == 0)].sort_values("seed")[metric].values
    imp = 100 * (b - a) / b
    half = sps.t.ppf(.975, len(imp) - 1) * imp.std(ddof=1) / np.sqrt(len(imp)) if len(imp) > 1 else np.nan
    sa = M[(M.rho == rho) & (M.policy == A) & (M.seg > 0)].sort_values(["seed", "seg"])[metric].values
    sb = M[(M.rho == rho) & (M.policy == B) & (M.seg > 0)].sort_values(["seed", "seg"])[metric].values
    return dict(rho=rho, A=A, B=B, metric=metric, improvement_pct=imp.mean(), CI95_halfwidth=half,
                ci_excludes_0=bool(imp.mean() - half > 0), seg_win_rate=float((sa < sb).mean()))

PAIRS = [("ADAPTIVE", "STATIC"), ("ADAPTIVE", "BEST_FIXED"), ("ADAPTIVE", "FEEDBACK"), ("ADAPTIVE", "ADAPTIVE_CONST"),
         ("ADAPTIVE", "ADAPTIVE_SHUFFLED"), ("ADAPTIVE", "ADAPTIVE_FROZEN"), ("FEEDBACK", "BEST_FIXED"), ("BEST_FIXED", "STATIC"), ("ORACLE", "ADAPTIVE")]
METRICS = ["high_p95", "macro_p95", "offload", "all_mean", "low_p95"]   # all: LOWER is better ("offload" = share of payload kept off-chain)
CMP = pd.DataFrame([compare(A, B, met, rho) for rho in RHOS for (A, B) in PAIRS
                    for met in METRICS])
CMP.to_csv(f"{OUT_DIR}/table_pairwise_comparisons.csv", index=False)
print("Positive improvement = first policy better (lower latency).  CI = t-interval over seeds.  seg_win_rate = share of (seed, time-segment) cells won.\n")
for met in METRICS:
    print(f"########## metric: {met}  (positive % = first policy better) ##########")
    display(CMP[CMP.metric == met].pivot_table(index=["A", "B"], columns="rho", values="improvement_pct").round(2))
print("########## minimum class SLA hit-rate (constraint: >= %.2f) ##########" % SLA_FLOOR)
display(M[M.seg == 0].groupby(["policy", "rho"])["sla_min"].mean().unstack("rho").loc[POLICIES].round(3))
print("########## share of time with backlog > 5 block-capacities (instability indicator) ##########")
display(M[M.seg == 0].groupby(["policy", "rho"])["frac_time_overloaded"].mean().unstack("rho").loc[POLICIES].round(3))
print("########## off-chain share of payload (lower = more data stays on-chain) ##########")
display(M[M.seg == 0].groupby(["policy", "rho"])["offload"].mean().unstack("rho").loc[POLICIES].round(3))

In [ ]:
# 12) FIGURES
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (cname, ci) in zip(axes, [("HIGH", 0), ("NORMAL", 1), ("LOW", 2)]):
    for pol, st in [("STATIC", "-"), ("BEST_FIXED", ":"), ("FEEDBACK", "--"), ("ADAPTIVE", "-")]:
        r = keep[pol]; m = (r["arr_t"] >= r["t_warm"]) & (r["cls"] == ci); x = np.sort(r["lat"][m])
        ax.plot(x, np.arange(1, len(x) + 1) / len(x), st, label=pol)
    ax.set_xscale("log"); ax.set_title(f"{cname} priority"); ax.set_xlabel("latency (s, log)"); ax.set_ylabel("CDF (confirmed tx)")
    ax.axvline(SIM["SLA"], color="gray", ls=":"); ax.legend()
plt.suptitle(f"Latency CDF by class (rho={rho_mid}, seed 0)"); plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_latency_cdf_by_class.png", dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(11, 3.8))
for pol in ["STATIC", "BEST_FIXED", "FEEDBACK", "ADAPTIVE"]:
    ax.plot(pd.Series(keep[pol]["backlog"]).rolling(144).mean().values, label=pol, lw=1)
ax.set_xlabel("block index (test)"); ax.set_ylabel("backlog (tx, 144-block mean)"); ax.set_title(f"Backlog over time (rho={rho_mid})")
ax.legend(); plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_backlog.png", dpi=200); plt.show()

seg_tab = M[(M.rho == rho_mid) & (M.seg > 0)].groupby(["policy", "seg"])["macro_p95"].mean().unstack("seg").loc[["STATIC","BEST_FIXED","FEEDBACK","ADAPTIVE","ORACLE"]]
ax = seg_tab.T.plot(kind="bar", figsize=(10, 4)); ax.set_ylabel("macro p95 latency (s)"); ax.set_xlabel("consecutive time segment of the test period")
ax.set_title(f"Consistency over time (rho={rho_mid})"); plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_segments.png", dpi=200); plt.show()

# Pareto view: off-chain share (x) vs class-averaged p95 latency (y).  Lower-left is better.
fig, axes = plt.subplots(1, len(RHOS), figsize=(4.2 * len(RHOS), 4), squeeze=False)
for ax, rho in zip(axes[0], RHOS):
    d_ = M[(M.rho == rho) & (M.seg == 0) & (~M.policy.isin(["STATIC", "PRIO_ONLY"]))].groupby("policy")[["offload", "macro_p95"]].mean()
    ax.scatter(d_.offload, d_.macro_p95)
    for pol, r_ in d_.iterrows(): ax.annotate(pol.replace("ADAPTIVE", "ADAPT"), (r_.offload, r_.macro_p95), fontsize=7)
    ax.set_title(f"rho={rho}"); ax.set_xlabel("off-chain share"); ax.set_ylabel("macro p95 (s)")
plt.suptitle("Service vs on-chain retention (STATIC / PRIO_ONLY omitted: unstable at high load)"); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig_pareto.png", dpi=200); plt.show()

In [ ]:
# 13) SENSITIVITY TO THE SIMULATOR'S ASSUMPTIONS (controllers keep the validation-tuned settings; no re-tuning; STATIC omitted: unstable)
variants = [(f"offloadable={o}, anchor=1.0", dict(OFFLOADABLE_P=o, ANCHOR=1.0)) for o in [0.2, 0.5, 0.8]]
variants += [("offloadable=0.5, anchor=0.5", dict(ANCHOR=0.5)), ("offloadable=0.5, anchor=2.0", dict(ANCHOR=2.0)),
             ("class mix 30/50/20", dict(CLASS_P=[0.30, 0.50, 0.20])), ("class mix 5/35/60", dict(CLASS_P=[0.05, 0.35, 0.60]))]
SENS_POL = ["BEST_FIXED", "ADAPTIVE_CONST", "ADAPTIVE"]; rows = []
for vname, mod in variants:
    S2 = dict(SIM, **mod); acc = {p: [] for p in SENS_POL}
    for sd in range(SENS_SEEDS):
        arr = make_arrivals(W_TEST, rho_mid, SEED + 500 + sd, S2)
        for pol in SENS_POL:
            rws, g_ = summarize(simulate(arr, W_TEST, pol, rho_mid, S2, TUNED), S2)
            acc[pol].append((np.mean([r["p95"] for r in rws if r["cls"] in CLS]), 1 - g_["onchain_units_per_tx"], g_["completed_tps"]))
    mp = {p: np.mean([a[0] for a in v]) for p, v in acc.items()}; of = {p: np.mean([a[1] for a in v]) for p, v in acc.items()}; ct = {p: np.mean([a[2] for a in v]) for p, v in acc.items()}
    rows.append(dict(variant=vname, ADAPT_vs_BESTFIXED_macro_p95_pct=100 * (mp["BEST_FIXED"] - mp["ADAPTIVE"]) / mp["BEST_FIXED"],
                     ADAPT_vs_BESTFIXED_offload_pct=100 * (of["BEST_FIXED"] - of["ADAPTIVE"]) / of["BEST_FIXED"],
                     completed_tps_BEST_FIXED=ct["BEST_FIXED"], completed_tps_ADAPTIVE=ct["ADAPTIVE"], completed_tps_change_pct=100 * (ct["ADAPTIVE"] - ct["BEST_FIXED"]) / ct["BEST_FIXED"],
                     ADAPT_vs_CONST_macro_p95_pct=100 * (mp["ADAPTIVE_CONST"] - mp["ADAPTIVE"]) / mp["ADAPTIVE_CONST"]))
sens = pd.DataFrame(rows); display(sens.round(2)); sens.to_csv(f"{OUT_DIR}/table_sensitivity_simulator.csv", index=False)
print("Positive = ADAPTIVE better (lower latency / less off-chain).  Report MIN and MAX of each column in the paper.")

In [ ]:
# 14) MEASURED OVERHEAD OF THE COMPLETE PIPELINE
pipe_d = make_models()[TB["SEL"]]; pipe_d.fit(df[FEATURES_DEM].iloc[max(0, a_va - WINDOW):a_va], df[TARGET_DEM].iloc[max(0, a_va - WINDOW):a_va])
row1 = df[FEATURES_DEM].iloc[[i_va]]; R = 300
t0 = time.perf_counter(); [pipe_d.predict(row1) for _ in range(R)]; t_inf = (time.perf_counter() - t0) / R * 1e3
win = raw.sort_values("number").iloc[2000:2200]
t0 = time.perf_counter(); [build_base_features(win) for _ in range(50)]; t_feat = (time.perf_counter() - t0) / 50 * 1e3
t0 = time.perf_counter()
for _ in range(100000): decide("ADAPTIVE", 5, 0.5, None, W_TEST, 0.85, SIM, TUNED)
t_dec = (time.perf_counter() - t0) / 100000 * 1e3
rng = np.random.default_rng(0); nq = 300
qt, qc, qs, qo = np.sort(rng.random(nq) * 500), rng.integers(0, 3, nq), rng.lognormal(-.32, .8, nq), rng.random(nq) < .5
def bench(reg): 
    t0 = time.perf_counter()
    for _ in range(500): select_block(600.0, qt, qc, qs, qo, REG[reg], SIM)
    return (time.perf_counter() - t0) / 500 * 1e3
overhead = pd.DataFrame([
    dict(stage="Feature generation (pandas, 200-block window)", ms_per_decision=t_feat),
    dict(stage=f"Demand-forecast inference ({TB['SEL']}, 1 row)", ms_per_decision=t_inf),
    dict(stage="Regime decision (threshold comparisons only = the O(1) part)", ms_per_decision=t_dec),
    dict(stage="Queue selection, adaptive CONG regime (300 queued tx)", ms_per_decision=bench("CONG")),
    dict(stage="Queue selection, static FIFO (300 queued tx)", ms_per_decision=bench("STATIC")),
    dict(stage="Drift-triggered recalibration (total over whole test period, ms)", ms_per_decision=WF["drift"]["retrain_time"] * 1e3)])
display(overhead.round(4)); overhead.to_csv(f"{OUT_DIR}/table_overhead.csv", index=False)
print("Compare with the ~600 s mean block interval. O(1) applies ONLY to the regime-decision row.")

In [ ]:
# 15) CLAIMS SCORECARD -> what the paper is ALLOWED to say (derived automatically; rules are fixed BEFORE looking at results)
claims = []
print("HELD-OUT TEST period (re-used across design iterations; NOT an independent confirmation):", audit["test_range"], "| design hash:", CONFIG_HASH, "\n")
def add(text, ok, evidence): claims.append(dict(claim=text, verdict="SUPPORTED" if ok else "NOT SUPPORTED", evidence=evidence))

add("Data: full coverage 2020-01..2021-03, contiguous blocks", (not QUICK), f"{audit['blocks_fetched']} blocks, {audit['date_range']}")
add("Simulator passes all sanity tests", bool(sanity.passed.all()), f"{int(sanity.passed.sum())}/{len(sanity)} passed")
ps = PRED_STATS["A: interval"]; ps_best = ps[ps.vs == TA["best_base"]].iloc[0]
add("Block INTERVAL is predictable beyond a naive baseline (Task A)", bool(ps_best.significant),
    f"{TA['SEL']} vs {TA['best_base']}: {ps_best.MAE_improvement_pct:+.2f}% CI[{ps_best.CI95_low:+.2f},{ps_best.CI95_high:+.2f}]  (expected: not predictable)")
pb = PRED_STATS["B: demand"]; pb_pers = pb[pb.vs == "Baseline: persistence (5-block mean)"].iloc[0]; pb_best = pb[pb.vs == TB["best_base"]].iloc[0]
add("Demand forecaster beats persistence (Task B)", bool(pb_pers.significant), f"{pb_pers.MAE_improvement_pct:+.2f}% CI[{pb_pers.CI95_low:+.2f},{pb_pers.CI95_high:+.2f}]")
add("Demand forecaster beats best validation baseline (Task B)", bool(pb_best.significant), f"vs {TB['best_base']}: {pb_best.MAE_improvement_pct:+.2f}% CI[{pb_best.CI95_low:+.2f},{pb_best.CI95_high:+.2f}]")
e_fro = np.abs(TB["y_te"] - WF["frozen"]["preds"]); e_dr = np.abs(TB["y_te"] - WF["drift"]["preds"]); lo, hi = block_bootstrap_ci(e_fro, e_dr)
add("Closed loop improves the forecaster on the REAL held-out test period (2021-Q1)", bool(lo > 0 and len(WF["drift"]["events"]) > 0),
    f"{len(WF['drift']['events'])} recalibrations; MAE change CI[{lo:+.2f},{hi:+.2f}]%  (a benefit is only expected if real drift occurs)")
det = stress[stress["mode"] == "drift"].iloc[0]
det_ok = (det.first_detection_cycle is not None) and (not pd.isna(det.first_detection_cycle))
add("Closed loop DETECTS injected drift and recovers (stress test, synthetic)", bool(det_ok and STRESS_CI[1] > 0),
    f"detection at cycle {det.first_detection_cycle} (drift starts {det.drift_starts_cycle}); post-drift MAE change {STRESS_CI[0]:+.1f}% CI[{STRESS_CI[1]:+.1f},{STRESS_CI[2]:+.1f}]")

def pair_rows(A, B, met): return CMP[(CMP.A == A) & (CMP.B == B) & (CMP.metric == met)]
def fmt(sub): return "; ".join(f"rho={r.rho}: {r.improvement_pct:+.1f}% [±{r.CI95_halfwidth:.1f}], seg-wins {r.seg_win_rate:.0%}" for r in sub.itertuples())
def superior(A, B, met, label):
    sub = pair_rows(A, B, met); add(label, bool(sub.ci_excludes_0.all()), fmt(sub))
def noninferior(A, B, met, label, tol=5.0):
    sub = pair_rows(A, B, met); add(label, bool(((sub.improvement_pct - sub.CI95_halfwidth) > -tol).all()), fmt(sub) + f"  (non-inferior if lower CI bound > -{tol}%)")

# stability of the executed baseline (reported as stability, NOT as a headline "improvement")
st = M[(M.seg == 0) & (M.policy == "STATIC")].groupby("rho").agg(sla=("sla_min", "mean"), over=("frac_time_overloaded", "mean"))
add("STATIC baseline is stable (min-class SLA >= 90 %) at every load", bool((st.sla >= SLA_FLOOR).all()),
    "; ".join(f"rho={k}: SLA {r.sla:.2f}, time-overloaded {r.over:.0%}" for k, r in st.iterrows()))
noninferior("ADAPTIVE", "BEST_FIXED", "high_p95", "ADAPTIVE keeps HIGH-priority p95 within 5 % of BEST_FIXED (non-inferiority)")
noninferior("ADAPTIVE", "BEST_FIXED", "macro_p95", "ADAPTIVE keeps class-averaged p95 within 5 % of BEST_FIXED (non-inferiority)")
superior("ADAPTIVE", "BEST_FIXED", "offload", "ADAPTIVE keeps MORE payload on-chain than BEST_FIXED (lower off-chain share, ALL loads)")
sg = M[(M.seg == 0) & (M.policy.isin(["ADAPTIVE", "BEST_FIXED"]))].pivot_table(index=["rho", "seed"], columns="policy", values="sla_min")
gap = (sg["ADAPTIVE"] - sg["BEST_FIXED"]).groupby("rho").mean()
add("ADAPTIVE keeps class SLA within 1 percentage point of BEST_FIXED (all loads)", bool((gap > -0.01).all()), "; ".join(f"rho={k}: {v*100:+.2f} pp" for k, v in gap.items()))
superior("ADAPTIVE", "ADAPTIVE_CONST", "macro_p95", "FORECAST VALUE: ADAPTIVE beats the no-forecast ablation on class-averaged p95 (ALL loads)")
superior("ADAPTIVE", "ADAPTIVE_CONST", "offload", "FORECAST VALUE: ADAPTIVE uses less off-chain than the no-forecast ablation (ALL loads)")
superior("ADAPTIVE", "ADAPTIVE_SHUFFLED", "macro_p95", "Forecast TIMING matters (beats shuffled-forecast placebo, class-averaged p95)")
superior("ADAPTIVE", "FEEDBACK", "macro_p95", "ADAPTIVE beats backlog-only FEEDBACK (class-averaged p95)")
superior("ADAPTIVE", "ADAPTIVE_FROZEN", "macro_p95", "Recalibrated forecaster beats frozen forecaster end-to-end (class-averaged p95)")
low = pair_rows("ADAPTIVE", "BEST_FIXED", "low_p95")
add("LOW-priority class is not sacrificed vs BEST_FIXED (p95 not >5 % worse at any load)", bool(((low.improvement_pct - low.CI95_halfwidth) > -5).all()), fmt(low))
gap_o = pair_rows("ORACLE", "ADAPTIVE", "macro_p95")
add("Gap to ORACLE (perfect arrival knowledge) is small (<5 % class-averaged p95)", bool((gap_o.improvement_pct < 5).all()), "; ".join(f"rho={r.rho}: {r.improvement_pct:+.1f}%" for r in gap_o.itertuples()))
add("Computational overhead negligible vs block interval (<1 % of 600 s)", bool(overhead.ms_per_decision.iloc[:4].sum() < 6000), f"{overhead.ms_per_decision.iloc[:4].sum():.2f} ms per decision")
sc = pd.DataFrame(claims); pd.set_option("display.max_colwidth", 220)
for r in sc.itertuples(): print(f"[{r.verdict:13s}] {r.claim}\n                 {r.evidence}\n")
sc.to_csv(f"{OUT_DIR}/CLAIMS_SCORECARD.csv", index=False)
print("RULE: write in the paper ONLY the claims marked SUPPORTED. Report NOT SUPPORTED items as honest negative findings / limitations.")

In [ ]:
# 16) SAVE EVERYTHING
summary = dict(config_hash=CONFIG_HASH, dataset=audit, taskA_selected=TA["SEL"], taskB_selected=TB["SEL"], tuned=TUNED, simulator_assumptions={k: (v.tolist() if hasattr(v, "tolist") else v) for k, v in SIM.items()},
               loads=RHOS, n_seeds=N_SEEDS, closed_loop=wf_table.round(4).to_dict("records"), overhead_ms=overhead.round(4).to_dict("records"),
               scorecard=sc.to_dict("records"))
json.dump(summary, open(f"{OUT_DIR}/RESULTS_SUMMARY.json", "w"), indent=2, default=str)
import shutil; shutil.make_archive("/content/results_v5_2_1", "zip", OUT_DIR)
print("Saved:", sorted(os.listdir(OUT_DIR)))
try:
    from google.colab import files; files.download("/content/results_v5_2_1.zip")
except Exception: pass